# Complex Quest Decomposition & Party Execution

This example demonstrates how the guild handles complex quests by:
1. Decomposing them into subtasks
2. Forming a party and distributing subtasks across adventurers
3. Running subtasks concurrently where possible
4. Evaluating combined results

**Prerequisites:** `pip install guildmaster-ai[openrouter]` and set `OPENROUTER_API_KEY` in your environment or `.env` file.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Build a guild with multiple adventurer types

In [ ]:
from guildmaster_ai import GeneralAdventurer, GuildBuilder
from guildmaster_ai.weapons.web_search import WebSearchWeapon

# Create a research-oriented adventurer with web search
researcher = GeneralAdventurer(name="Researcher")
researcher.equip_weapon(WebSearchWeapon())

guild = (
    GuildBuilder()
    .with_llm_provider("openrouter")
    .register_adventurer(GeneralAdventurer, count=2)  # 2 general adventurers
    .register_adventurer(researcher)  # 1 research specialist
    .build()
)

print(f"Guild roster: {len(guild.roster)} adventurers")
for adv in guild.roster:
    print(f"  - {adv.name or adv.id}: talents={adv.talents}, weapons={adv.weapons}")

## Post a complex quest

When a quest is complex enough, the Guildmaster will automatically decompose it into subtasks.

In [ ]:
result = await guild.run_quest(
    "Create a comprehensive comparison of Python web frameworks. "
    "Cover Django, FastAPI, and Flask. For each framework, describe "
    "its architecture, performance characteristics, and best use cases. "
    "Then provide a summary table comparing all three."
)

print(f"Quest success: {result.success}")
print(f"Subtask count: {result.data.get('subtask_count', 'N/A (simple quest)')}")
print(f"\nResult summary:\n{result.summary[:500]}...")

## Inspect the quest lifecycle

In [ ]:
# Get the parent quest
parent = guild.get_quest(result.quest_id)
print(f"Quest: {parent.title}")
print(f"Composite: {parent.is_composite}")
print(f"Status: {parent.status}")
print(f"\nHistory ({len(parent.history)} events):")
for entry in parent.history:
    print(f"  [{entry.event_type}] {entry.actor}: {entry.payload}")

In [ ]:
# View subtask quests
subtasks = [q for q in guild.quests if q.parent_quest_id == result.quest_id]
print(f"Subtasks: {len(subtasks)}")
for st in sorted(subtasks, key=lambda q: q.subtask_index or 0):
    st_result = guild.get_result(st.id)
    status = "SUCCESS" if st_result and st_result.success else "FAILED"
    print(f"  [{st.subtask_index}] {st.title} — {status}")
    if st_result:
        print(f"      {st_result.summary[:100]}...")

## Guild state after complex quest

In [ ]:
info = guild.info
print(f"Total quests: {info.total_quests} (including subtasks)")
print(f"Completed: {info.completed}")
print(f"Archived: {info.archived}")
print(f"Failed: {info.failed}")